# Two-threshold structural robustness

This notebook closes the two-threshold phase (Chapter 10, gate G6). It asks which two-clock conclusions survive a change of **symbol**, **absolute threshold size**, and **threshold ratio**, and which are artifacts of one market or one pair.

The earlier labs deliberately used different settings:

- `2_threshold_dc.ipynb` (Chapters 6, 7 and 9) uses **EURUSD at 0.20% / 0.40%**;
- `2_threshold_dc_event_characteristics.ipynb` (Chapters 8 and 9) uses **USDJPY at 0.25% / 0.50%**.

Read separately, a difference between those notebooks mixes a symbol change with a threshold change. Here both symbols and every pair run through one fixed pipeline on one common calendar window, so each comparison changes exactly one thing.

The cases are declared in the next cell before any result is computed. They are not searched or ranked by propagation rate. Propagated/rejected remains a hindsight label until its resolving DCC; every live feature is measured on its own observation bar.

This is structural reproduction. It is not an optimised threshold selection, not untouched out-of-sample confirmation, and not evidence of trading profitability.

In [ ]:
from enum import StrEnum
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score

from dc_basics.core import dc_live_features, parse_dc_events
from dc_basics.data import NATIVE_BAR_MINUTES, load_close, resolve_data_root
from dc_basics.enums import AttemptOutcome, DcMode
from dc_basics.events import prepare_event_ledger
from dc_basics.live import build_causal_running_extreme
from dc_basics.two_scale import build_counter_scale_trajectory_panel, build_disagreement_episodes

DATA_ROOT = resolve_data_root()
CHECKPOINT_BARS = [0, 5, 10, 20, 40, 80, 160]
DEVELOPED_CHECKPOINT = 20
MIN_AUC_CLASS_COUNT = 20
MIN_STRATUM_ATTEMPTS = 30
INITIATION_FEATURES = ["tmv", "t", "r"]
LIVE_FEATURES = [
    "small_tmv_live",
    "small_t_live",
    "small_rdc_live",
    "post_dcc_excursion_theta",
    "max_post_dcc_excursion_theta",
    "large_dcc_progress",
]
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

In [ ]:
class Scale(StrEnum):
    SMALL = "small"
    LARGE = "large"


class Question(StrEnum):
    SYMBOL = "symbol: same pair, different market"
    ABSOLUTE_SCALE = "absolute scale: same market and ratio 2, different size"
    RATIO = "ratio: same market and small threshold, different ratio"


class CaseId(StrEnum):
    JPY_15_30 = "USDJPY 15/30"
    JPY_20_40 = "USDJPY 20/40"
    JPY_25_50 = "USDJPY 25/50"
    JPY_20_30 = "USDJPY 20/30"
    EUR_15_30 = "EURUSD 15/30"
    EUR_20_40 = "EURUSD 20/40"
    EUR_25_50 = "EURUSD 25/50"
    EUR_20_30 = "EURUSD 20/30"


class CaseSpec(NamedTuple):
    case_id: CaseId
    symbol: str
    small_theta: float
    large_theta: float
    role: str


CASE_SPECS = (
    CaseSpec(CaseId.JPY_15_30, "USDJPY.r", 0.0015, 0.0030, "absolute-scale variant"),
    CaseSpec(CaseId.JPY_20_40, "USDJPY.r", 0.0020, 0.0040, "symbol contrast for the EURUSD lab pair"),
    CaseSpec(CaseId.JPY_25_50, "USDJPY.r", 0.0025, 0.0050, "pair used in 2_threshold_dc_event_characteristics"),
    CaseSpec(CaseId.JPY_20_30, "USDJPY.r", 0.0020, 0.0030, "ratio contrast (1.5 instead of 2)"),
    CaseSpec(CaseId.EUR_15_30, "EURUSD.r", 0.0015, 0.0030, "absolute-scale variant"),
    CaseSpec(CaseId.EUR_20_40, "EURUSD.r", 0.0020, 0.0040, "pair used in 2_threshold_dc"),
    CaseSpec(CaseId.EUR_25_50, "EURUSD.r", 0.0025, 0.0050, "symbol contrast for the USDJPY lab pair"),
    CaseSpec(CaseId.EUR_20_30, "EURUSD.r", 0.0020, 0.0030, "ratio contrast (1.5 instead of 2)"),
)
COMPARISONS = {
    Question.SYMBOL: [
        (CaseId.EUR_15_30, CaseId.JPY_15_30),
        (CaseId.EUR_20_40, CaseId.JPY_20_40),
        (CaseId.EUR_25_50, CaseId.JPY_25_50),
    ],
    Question.ABSOLUTE_SCALE: [
        (CaseId.EUR_15_30, CaseId.EUR_20_40, CaseId.EUR_25_50),
        (CaseId.JPY_15_30, CaseId.JPY_20_40, CaseId.JPY_25_50),
    ],
    Question.RATIO: [
        (CaseId.EUR_20_30, CaseId.EUR_20_40),
        (CaseId.JPY_20_30, CaseId.JPY_20_40),
    ],
}
SPEC_BY_ID = {spec.case_id: spec for spec in CASE_SPECS}
for question, groups in COMPARISONS.items():
    for group in groups:
        specs = [SPEC_BY_ID[case_id] for case_id in group]
        same_symbol = len({spec.symbol for spec in specs}) == 1
        same_pair = len({(spec.small_theta, spec.large_theta) for spec in specs}) == 1
        same_ratio = len({round(spec.large_theta / spec.small_theta, 12) for spec in specs}) == 1
        same_small = len({spec.small_theta for spec in specs}) == 1
        changes_one_thing = {
            Question.SYMBOL: same_pair and not same_symbol,
            Question.ABSOLUTE_SCALE: same_symbol and same_ratio and not same_pair,
            Question.RATIO: same_symbol and same_small and not same_ratio,
        }[question]
        if not changes_one_thing:
            raise AssertionError(f"{question.value}: {group} changes more than one thing.")

case_design = pd.DataFrame.from_records([
    {
        "case": spec.case_id.value,
        "symbol": spec.symbol,
        "small_theta": spec.small_theta,
        "large_theta": spec.large_theta,
        "ratio": spec.large_theta / spec.small_theta,
        "role": spec.role,
    }
    for spec in CASE_SPECS
]).set_index("case")
case_design.style.format({"small_theta": "{:.2%}", "large_theta": "{:.2%}", "ratio": "{:.2f}"})

## Comparison contract

Each comparison changes one design variable; the cell above refuses a grouping that changes two.

1. **Symbol:** EURUSD vs USDJPY at the same pair.
2. **Absolute scale:** 15/30, 20/40 and 25/50 bps within one symbol. The ratio stays 2, so the lower-scale contest needs the same relative move to reach the upper clock; only the size of a "meaningful" move changes.
3. **Ratio:** 20/30 vs 20/40 within one symbol. The small clock is identical; only the distance to the upper clock changes.

Exact counts, durations and propagation rates are expected to change. A conclusion counts as structural only if its direction and meaning recur in every adequately sampled case.

In [ ]:
symbols = tuple(dict.fromkeys(spec.symbol for spec in CASE_SPECS))
raw_prices = {symbol: load_close(DATA_ROOT, symbol) for symbol in symbols}
common_start = max(close.index.min() for close in raw_prices.values())
common_end = min(close.index.max() for close in raw_prices.values())
if common_start >= common_end:
    raise ValueError("The selected symbols have no common data interval.")

prices_by_symbol = {symbol: close.loc[common_start:common_end].copy() for symbol, close in raw_prices.items()}
coverage_summary = pd.DataFrame.from_records([
    {
        "symbol": symbol,
        "sampling_minutes": NATIVE_BAR_MINUTES,
        "bars": len(close),
        "start": close.index.min(),
        "end": close.index.max(),
        "bars_outside_common_window": len(raw_prices[symbol]) - len(close),
    }
    for symbol, close in prices_by_symbol.items()
]).set_index("symbol")
print(f"Common comparison window: {common_start} to {common_end}")
coverage_summary.style.format({"bars": "{:,}", "bars_outside_common_window": "{:,}"})

## Reconstruct both clocks and every disagreement episode

The definitions are exactly those of `2_threshold_dc.ipynb`:

- each threshold runs its own parser and live-mode clock on the same close series;
- completed TMV, T and R carry `information_time = dcc_idx`;
- an episode starts when a small DCC moves the small mode against the large mode known on the previous bar;
- it is `propagated` when a large DCC confirms the target direction first, `rejected` when the small clock reverses first, `ambiguous_same_bar` when both happen on one bar, and `unresolved_at_sample_end` (right-censored) when the data ends first.

The completed-leg/live-mode alignment and the nesting structure are audited for every case. Nesting parity (an odd number of small DCCs between consecutive large DCCs) is a near-invariant under multiplicative thresholds, not a law, so exceptions are counted rather than asserted.

In [ ]:
def build_two_clocks(close: pd.Series, spec: CaseSpec) -> tuple[dict, dict, pd.DataFrame]:
    ledgers = {}
    live_frames = {}
    mode_frame = pd.DataFrame(index=close.index)
    for scale, prefix, theta in (
        (Scale.SMALL, "S", spec.small_theta),
        (Scale.LARGE, "L", spec.large_theta),
    ):
        ledger = prepare_event_ledger(parse_dc_events(prices=close, theta=theta), prefix, scale, theta)
        live = dc_live_features(close, theta=theta)
        for completed_leg, dcc_flag in ((DcMode.UP, "downward_dcc"), (DcMode.DOWN, "upward_dcc")):
            event_times = pd.Index(ledger.loc[ledger["direction"].eq(completed_leg.value), "dcc_idx"])
            live_times = live.index[live[dcc_flag].astype(bool)]
            if len(event_times.symmetric_difference(live_times)):
                raise AssertionError(f"{spec.case_id.value} {scale.value}: completed {completed_leg.value} legs "
                                     f"do not align with live {dcc_flag} bars.")
        ledgers[scale] = ledger
        live_frames[scale] = live
        mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
        mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
        mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
        mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(
            ledger.set_index("dcc_idx")["event_id"]
        )
    return ledgers, live_frames, mode_frame


def audit_nesting(ledgers: dict) -> dict[str, int]:
    small_positions = ledgers[Scale.SMALL]["dcc_pos"].to_numpy(dtype=np.int64)
    large_positions = ledgers[Scale.LARGE]["dcc_pos"].to_numpy(dtype=np.int64)
    counts = (
        np.searchsorted(small_positions, large_positions[1:], side="right")
        - np.searchsorted(small_positions, large_positions[:-1], side="right")
    )
    return {
        "large_intervals": len(counts),
        "intervals_with_even_small_count": int((counts % 2 == 0).sum()),
        "intervals_without_small_dcc": int((counts == 0).sum()),
    }


def build_case(close: pd.Series, spec: CaseSpec) -> dict[str, object]:
    ledgers, live_frames, mode_frame = build_two_clocks(close, spec)
    episodes = build_disagreement_episodes(mode_frame, close)
    large_running_extreme = build_causal_running_extreme(close, live_frames[Scale.LARGE])
    trajectories, clean_attempts = build_counter_scale_trajectory_panel(
        episodes,
        close,
        live_frames[Scale.SMALL],
        mode_frame=mode_frame,
        large_running_extreme=large_running_extreme,
        small_theta=spec.small_theta,
        large_theta=spec.large_theta,
    )
    completed_small_legs = ledgers[Scale.SMALL][["event_id", "information_time", *INITIATION_FEATURES]].rename(
        columns={"event_id": "small_event_id", "information_time": "initiation_information_time"}
    )
    clean_attempts = clean_attempts.merge(
        completed_small_legs, on="small_event_id", how="left", validate="one_to_one"
    )
    if clean_attempts[INITIATION_FEATURES].isna().any().any():
        raise AssertionError(f"{spec.case_id.value}: an attempt has no completed small leg.")
    if not clean_attempts["initiation_information_time"].eq(clean_attempts["start_time"]).all():
        raise AssertionError(f"{spec.case_id.value}: initiation features are not timestamped at the seed.")
    return {
        "spec": spec,
        "prices": close,
        "ledgers": ledgers,
        "mode_frame": mode_frame,
        "episodes": episodes,
        "clean_attempts": clean_attempts,
        "trajectories": trajectories,
        "nesting": audit_nesting(ledgers),
    }

## Run the pre-registered cases

This is the expensive cell. No result from one case changes another case's configuration.

In [ ]:
case_results = {}
for spec in CASE_SPECS:
    print(f"Building {spec.case_id.value}...")
    case_results[spec.case_id] = build_case(prices_by_symbol[spec.symbol], spec)
print(f"Completed {len(case_results)} pre-registered cases.")

## Audit 1: counts and exclusions before any percentage

Every episode is accounted for. The columns reconcile exactly:

- `episodes = propagated + rejected + ambiguous_same_bar + unresolved_at_sample_end`;
- `clean_binary_attempts = propagated + rejected - same_bar_propagations`.

A same-bar propagation (seed and large DCC on one bar) has no unresolved life, so it is excluded from initiation-overlap and live-development diagnostics but **kept** in the counts. A rejection always needs a later small DCC, so same-bar resolutions can only be propagations. Unresolved episodes are right-censored by the end of the sample.

In [ ]:
count_records = []
for case_id, result in case_results.items():
    spec = result["spec"]
    episodes = result["episodes"]
    outcomes = episodes["outcome"].value_counts()
    same_bar = episodes["resolution_bars"].eq(0).fillna(False)
    same_bar_propagations = int((same_bar & episodes["outcome"].eq(AttemptOutcome.PROPAGATED.value)).sum())
    if int(same_bar.sum()) != same_bar_propagations:
        raise AssertionError(f"{case_id.value}: a same-bar resolution is not a propagation.")
    record = {
        "case": case_id.value,
        "symbol": spec.symbol,
        "pair": f"{spec.small_theta:.2%} / {spec.large_theta:.2%}",
        "bars": len(result["prices"]),
        "small_events": len(result["ledgers"][Scale.SMALL]),
        "large_events": len(result["ledgers"][Scale.LARGE]),
        **result["nesting"],
        "episodes": len(episodes),
        "propagated": int(outcomes.get(AttemptOutcome.PROPAGATED.value, 0)),
        "rejected": int(outcomes.get(AttemptOutcome.REJECTED.value, 0)),
        "ambiguous_same_bar": int(outcomes.get(AttemptOutcome.AMBIGUOUS.value, 0)),
        "unresolved_at_sample_end": int(outcomes.get(AttemptOutcome.UNRESOLVED.value, 0)),
        "same_bar_propagations": same_bar_propagations,
        "clean_binary_attempts": len(result["clean_attempts"]),
    }
    if record["episodes"] != (
        record["propagated"] + record["rejected"] + record["ambiguous_same_bar"] + record["unresolved_at_sample_end"]
    ):
        raise AssertionError(f"{case_id.value}: episode outcomes do not reconcile.")
    if record["clean_binary_attempts"] != record["propagated"] + record["rejected"] - same_bar_propagations:
        raise AssertionError(f"{case_id.value}: clean attempts do not reconcile.")
    count_records.append(record)

count_summary = pd.DataFrame.from_records(count_records).set_index("case")
count_summary.style.format({
    column: "{:,}" for column in count_summary.columns if column not in ("symbol", "pair")
})

## Audit 2: propagation rates, and why they follow from event counts

Two rates are reported and labelled, because they answer different questions:

- `rate_all_resolved` = propagated / (propagated + rejected), including same-bar propagations;
- `rate_clean` = the same ratio among clean binary attempts (same-bar propagations removed). It is lower by construction, because only propagations can be same-bar.

**Arithmetic check.** Between two consecutive large DCCs the small clock fires an odd number $n$ of times (nesting parity). Those firings alternate seed, rejection, seed, …, seed, and the last seed propagates. So each large interval holds $(n+1)/2$ attempts and exactly one propagation. Summing over intervals, attempts $\approx (N_S + N_L)/2$ and propagations $\approx N_L$, giving

$$\text{rate} \approx \frac{2}{N_S/N_L + 1}.$$

The only slack comes from the sample edges and from nesting-parity exceptions (Audit 1), so when both are absent the "approximation" is exact. A propagation rate that matches it is an arithmetic consequence of how many events each clock produces, not a separate behavioural fact about the market. Before reading the table, predict which comparison should move the rate most: the event-count ratio $N_S/N_L$ is driven mainly by the threshold **ratio**, not by the absolute size of the pair.

In [ ]:
rate_records = []
for case_id, result in case_results.items():
    spec = result["spec"]
    counts = count_summary.loc[case_id.value]
    resolved = counts["propagated"] + counts["rejected"]
    clean = result["clean_attempts"]
    event_ratio = counts["small_events"] / counts["large_events"]
    rate_records.append({
        "case": case_id.value,
        "ratio": spec.large_theta / spec.small_theta,
        "event_count_ratio": event_ratio,
        "resolved_attempts": resolved,
        "rate_all_resolved": counts["propagated"] / resolved,
        "rate_from_event_counts": 2.0 / (event_ratio + 1.0),
        "clean_binary_attempts": len(clean),
        "rate_clean": clean["future_outcome"].eq(AttemptOutcome.PROPAGATED.value).mean(),
        "median_bars_propagated": clean.loc[
            clean["future_outcome"].eq(AttemptOutcome.PROPAGATED.value), "resolution_bars"
        ].median(),
        "median_bars_rejected": clean.loc[
            clean["future_outcome"].eq(AttemptOutcome.REJECTED.value), "resolution_bars"
        ].median(),
    })
rate_summary = pd.DataFrame.from_records(rate_records).set_index("case")
rate_summary["arithmetic_gap"] = rate_summary["rate_all_resolved"] - rate_summary["rate_from_event_counts"]
rate_summary.style.format({
    "ratio": "{:.2f}",
    "event_count_ratio": "{:.2f}",
    "resolved_attempts": "{:,.0f}",
    "rate_all_resolved": "{:.2%}",
    "rate_from_event_counts": "{:.2%}",
    "clean_binary_attempts": "{:,}",
    "rate_clean": "{:.2%}",
    "median_bars_propagated": "{:.0f}",
    "median_bars_rejected": "{:.0f}",
    "arithmetic_gap": "{:+.2%}",
})

## Audit 3: does initiation geometry overlap across outcomes?

Chapter 8's claim is that the completed small leg (TMV, T, R known at the seed) barely separates later propagation from rejection. For each case, the clean binary attempts are grouped by their hindsight outcome and each completed-leg feature gets an orientation-free AUC (0.5 = no ordering, 1.0 = perfect ordering). AUC is an in-sample overlap description, not a forecast. Cells with fewer than the minimum number of propagated or rejected attempts are left empty.

In [ ]:
def orientation_free_auc(target: pd.Series, values: pd.Series) -> tuple[float, str]:
    values = values.replace([np.inf, -np.inf], np.nan)
    valid = values.notna()
    target = target.loc[valid]
    values = values.loc[valid]
    positives = int(target.sum())
    negatives = int(len(target) - positives)
    if positives < MIN_AUC_CLASS_COUNT or negatives < MIN_AUC_CLASS_COUNT:
        return np.nan, "insufficient_sample"
    if values.nunique() < 2:
        return np.nan, "constant_feature"
    raw_auc = roc_auc_score(target, values)
    return max(raw_auc, 1.0 - raw_auc), ("higher" if raw_auc >= 0.5 else "lower")


initiation_records = []
for case_id, result in case_results.items():
    clean = result["clean_attempts"]
    target = clean["future_outcome"].eq(AttemptOutcome.PROPAGATED.value).astype(np.int8)
    for feature in INITIATION_FEATURES:
        auc, direction = orientation_free_auc(target, clean[feature])
        initiation_records.append({
            "case": case_id.value,
            "feature": feature,
            "propagated": int(target.sum()),
            "rejected": int(len(target) - target.sum()),
            "orientation_free_auc": auc,
            "higher_or_lower_value_with_propagation": direction,
        })
initiation_overlap = pd.DataFrame.from_records(initiation_records)
initiation_overlap.pivot(index="case", columns="feature", values="orientation_free_auc").loc[
    [case_id.value for case_id in case_results]
].style.format("{:.3f}")

## Audit 4: live development without confusing it with survivor selection

At checkpoint $k$ only attempts still unresolved after $k$ bars remain. Comparing their AUC at $k$ with the **whole** cohort's AUC at bar 0 mixes two effects: information gained by watching the path, and the change in who is left. This audit separates them by holding the cohort fixed:

- `auc_at_k`: the live feature measured at bar $k$ for the survivors to $k$;
- `auc_same_cohort_at_0`: the same feature for **the same survivors**, measured at their seed bar;
- `information_gain = auc_at_k - auc_same_cohort_at_0`: what waiting added for that fixed cohort;
- `auc_completed_leg_same_cohort`: the best completed-leg feature (TMV, T or R) for the same survivors.

Survivor counts and the later propagation rate among survivors are shown beside every row; a falling rate among survivors is a population statement, not evidence that waiting causes failure.

In [ ]:
def checkpoint_records(case_id: CaseId, result: dict[str, object], feature: str) -> list[dict[str, object]]:
    trajectories = result["trajectories"]
    clean = result["clean_attempts"].set_index("attempt_id")
    at_seed = trajectories.loc[trajectories["elapsed_bars"].eq(0)].set_index("attempt_id")
    records = []
    for elapsed_bars in CHECKPOINT_BARS:
        survivors = trajectories.loc[trajectories["elapsed_bars"].eq(elapsed_bars)].set_index("attempt_id")
        target = survivors["future_outcome"].eq(AttemptOutcome.PROPAGATED.value).astype(np.int8)
        auc_at_k, direction_at_k = orientation_free_auc(target, survivors[feature])
        auc_at_0, _ = orientation_free_auc(target, at_seed.loc[survivors.index, feature])
        completed_aucs = [
            orientation_free_auc(target, clean.loc[survivors.index, column])[0] for column in INITIATION_FEATURES
        ]
        records.append({
            "case": case_id.value,
            "feature": feature,
            "elapsed_bars": elapsed_bars,
            "survivors": len(survivors),
            "survivor_share": len(survivors) / len(clean),
            "later_propagation_rate": target.mean() if len(target) else np.nan,
            "auc_at_k": auc_at_k,
            "higher_or_lower_value_with_propagation": direction_at_k,
            "auc_same_cohort_at_0": auc_at_0,
            "information_gain": auc_at_k - auc_at_0,
            "auc_completed_leg_same_cohort": (
                np.nan if np.isnan(completed_aucs).all() else np.nanmax(completed_aucs)
            ),
        })
    return records


live_records = []
for case_id, result in case_results.items():
    for feature in ("large_dcc_progress", "max_post_dcc_excursion_theta"):
        live_records.extend(checkpoint_records(case_id, result, feature))
live_development = pd.DataFrame.from_records(live_records)
live_development.loc[live_development["feature"].eq("large_dcc_progress")].set_index(
    ["case", "elapsed_bars"]
).drop(columns="feature").style.format({
    "survivors": "{:,}",
    "survivor_share": "{:.1%}",
    "later_propagation_rate": "{:.2%}",
    "auc_at_k": "{:.3f}",
    "auc_same_cohort_at_0": "{:.3f}",
    "information_gain": "{:+.3f}",
    "auc_completed_leg_same_cohort": "{:.3f}",
})

In [ ]:
progress_rows = live_development.loc[live_development["feature"].eq("large_dcc_progress")]
case_colors = dict(zip(
    [spec.case_id.value for spec in CASE_SPECS],
    ["#16a34a", "#2563eb", "#7c3aed", "#0f766e", "#f59e0b", "#dc2626", "#db2777", "#64748b"],
    strict=True,
))
development_figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Survivors at bar k, measured at bar k", "Same survivors, measured at their seed bar"],
)
for column_number, auc_column in enumerate(["auc_at_k", "auc_same_cohort_at_0"], start=1):
    for case, case_rows in progress_rows.groupby("case", sort=False):
        development_figure.add_trace(go.Scatter(
            x=case_rows["elapsed_bars"],
            y=case_rows[auc_column],
            mode="lines+markers",
            name=case,
            legendgroup=case,
            showlegend=column_number == 1,
            line={"color": case_colors[case], "width": 2},
            customdata=case_rows[["survivors", "later_propagation_rate"]].to_numpy(),
            hovertemplate=(
                "case=%{fullData.name}<br>k=%{x}<br>AUC=%{y:.3f}<br>survivors=%{customdata[0]:,.0f}"
                "<br>later propagation=%{customdata[1]:.2%}<extra></extra>"
            ),
        ), row=1, col=column_number)
    development_figure.add_hline(y=0.5, line={"color": "#64748b", "dash": "dot"}, row=1, col=column_number)
    development_figure.update_xaxes(title_text="Checkpoint k (bars after the seed)", row=1, col=column_number)
    development_figure.update_yaxes(range=[0.45, 1.0], row=1, col=column_number)
development_figure.update_layout(
    title="Progress toward the large DCC: separation of later outcomes (hindsight grouping, fixed cohorts)",
    template="plotly_white",
    height=520,
    legend={"orientation": "h", "y": 1.15, "x": 0.0},
)
development_figure.show()

## Audit 5: direction and calendar-period concentration

A relationship carried by one direction or one period is weaker evidence of structure. Periods are **calendar** thirds of the common window, so every case splits on the same dates. Strata with fewer than `MIN_STRATUM_ATTEMPTS` clean attempts are flagged and excluded from the range columns.

In [ ]:
period_edges = pd.date_range(common_start, common_end, periods=4)
period_labels = ["early", "middle", "late"]
stratum_records = []
for case_id, result in case_results.items():
    episodes = result["episodes"]
    clean = result["clean_attempts"].copy()
    clean["period"] = pd.cut(clean["start_time"], bins=period_edges, labels=period_labels, include_lowest=True)
    resolved = episodes.loc[episodes["outcome"].isin([
        AttemptOutcome.PROPAGATED.value, AttemptOutcome.REJECTED.value
    ])].copy()
    resolved["period"] = pd.cut(resolved["start_time"], bins=period_edges, labels=period_labels, include_lowest=True)
    for stratum_kind, column in (("direction", "target_small_mode"), ("period", "period")):
        for stratum, group in resolved.groupby(column, sort=True, observed=True):
            clean_group = clean.loc[clean[column].eq(stratum)]
            stratum_records.append({
                "case": case_id.value,
                "stratum_kind": stratum_kind,
                "stratum": str(stratum),
                "resolved_attempts": len(group),
                "rate_all_resolved": group["outcome"].eq(AttemptOutcome.PROPAGATED.value).mean(),
                "clean_binary_attempts": len(clean_group),
                "adequate": len(clean_group) >= MIN_STRATUM_ATTEMPTS,
            })
stratum_summary = pd.DataFrame.from_records(stratum_records)
display(stratum_summary.set_index(["case", "stratum_kind", "stratum"]).style.format({
    "resolved_attempts": "{:,}",
    "rate_all_resolved": "{:.2%}",
    "clean_binary_attempts": "{:,}",
}))

adequate_strata = stratum_summary.loc[stratum_summary["adequate"]]
stratum_ranges = adequate_strata.groupby(["case", "stratum_kind"])["rate_all_resolved"].agg(
    adequate_strata="size", rate_min="min", rate_max="max"
).unstack("stratum_kind")
stratum_ranges

## Audit 6: interpretation matrix

One row per case. Each column is one Chapter 10 conclusion, evaluated only where the sample is adequate:

- `initiation_overlap`: the best completed-leg AUC stays modest (below 0.60);
- `waiting_adds_information`: for the cohort surviving to `DEVELOPED_CHECKPOINT` bars, progress at that checkpoint separates outcomes better than the same cohort's progress at its seed;
- `higher_progress_with_propagation`: the associated direction at that checkpoint is "higher";
- `rate_matches_event_counts`: the observed rate is within 2 percentage points of the event-count arithmetic;
- `survivor_rate_falls`: the later propagation rate among survivors is lower at `DEVELOPED_CHECKPOINT` than at the seed.

A `False` is evidence against generality, not a reason to drop or retune that case.

In [ ]:
INITIATION_OVERLAP_AUC = 0.60
ARITHMETIC_TOLERANCE = 0.02

progress_by_case = progress_rows.set_index(["case", "elapsed_bars"])
interpretation_records = []
for case_id in case_results:
    case = case_id.value
    initiation = initiation_overlap.loc[initiation_overlap["case"].eq(case)]
    developed = progress_by_case.loc[(case, DEVELOPED_CHECKPOINT)]
    at_seed = progress_by_case.loc[(case, 0)]
    adequate = bool(np.isfinite(developed["auc_at_k"]) and np.isfinite(developed["auc_same_cohort_at_0"]))
    best_initiation_auc = initiation["orientation_free_auc"].max()
    interpretation_records.append({
        "case": case,
        "adequate_developed_sample": adequate,
        "best_initiation_auc": best_initiation_auc,
        "initiation_overlap": bool(best_initiation_auc < INITIATION_OVERLAP_AUC) if np.isfinite(best_initiation_auc) else pd.NA,
        "information_gain_at_developed": developed["information_gain"],
        "waiting_adds_information": bool(developed["information_gain"] > 0.0) if adequate else pd.NA,
        "higher_progress_with_propagation": (
            developed["higher_or_lower_value_with_propagation"] == "higher" if adequate else pd.NA
        ),
        "rate_matches_event_counts": bool(abs(rate_summary.loc[case, "arithmetic_gap"]) <= ARITHMETIC_TOLERANCE),
        "survivor_rate_falls": bool(developed["later_propagation_rate"] < at_seed["later_propagation_rate"]),
    })
interpretation_matrix = pd.DataFrame.from_records(interpretation_records).set_index("case")
display(interpretation_matrix.style.format({
    "best_initiation_auc": "{:.3f}",
    "information_gain_at_developed": "{:+.3f}",
}))

conclusion_columns = [
    "initiation_overlap",
    "waiting_adds_information",
    "higher_progress_with_propagation",
    "rate_matches_event_counts",
    "survivor_rate_falls",
]
recurrence_summary = pd.DataFrame({
    "evaluable_cases": interpretation_matrix[conclusion_columns].notna().sum(),
    "cases_where_true": interpretation_matrix[conclusion_columns].fillna(False).astype(bool).sum(),
})
recurrence_summary

## Case record

Chapter 10 asks for one record per case: symbol, interval, sampling frequency, pair, event counts, attempt counts, exclusions, and direction-specific results. This table is that record. Keep it with your interpretation notes.

In [ ]:
direction_rates = stratum_summary.loc[stratum_summary["stratum_kind"].eq("direction")].pivot(
    index="case", columns="stratum", values="rate_all_resolved"
).add_prefix("rate_target_")
case_record = (
    count_summary[[
        "symbol", "pair", "bars", "small_events", "large_events", "episodes", "propagated", "rejected",
        "ambiguous_same_bar", "unresolved_at_sample_end", "same_bar_propagations", "clean_binary_attempts",
        "intervals_with_even_small_count",
    ]]
    .join(rate_summary[["rate_all_resolved", "rate_clean"]])
    .join(direction_rates)
)
case_record.insert(1, "sampling_minutes", NATIVE_BAR_MINUTES)
case_record.insert(2, "interval", f"{common_start:%Y-%m-%d} to {common_end:%Y-%m-%d}")
case_record.style.format({
    "bars": "{:,}",
    "small_events": "{:,}",
    "large_events": "{:,}",
    "episodes": "{:,}",
    "propagated": "{:,}",
    "rejected": "{:,}",
    "ambiguous_same_bar": "{:,}",
    "unresolved_at_sample_end": "{:,}",
    "same_bar_propagations": "{:,}",
    "clean_binary_attempts": "{:,}",
    "intervals_with_even_small_count": "{:,}",
    "rate_all_resolved": "{:.2%}",
    "rate_clean": "{:.2%}",
    "rate_target_down": "{:.2%}",
    "rate_target_up": "{:.2%}",
})

# Two-threshold robustness completion gate (G6)

Answer from the outputs above, in your own words, before starting Chapter 11:

1. Which conclusions recurred in every adequately sampled case, and which numbers changed?
2. Why did event counts and durations change with the absolute threshold, and why did the propagation rate barely move when the ratio stayed at 2?
3. Why did the ratio comparison (20/30 vs 20/40) move the propagation rate, and how does the event-count arithmetic in Audit 2 explain it?
4. How many episodes were same-bar, ambiguous or unresolved in each case, and why are they reported rather than silently dropped?
5. Did completed initiation geometry overlap across outcomes in every case?
6. For a fixed survivor cohort, did watching the path add separation, or did the apparent gain come from survivor selection?
7. Is any relationship concentrated in one direction or one calendar third?
8. Why can a sparse case neither confirm nor refute a relationship?
9. Why is recurring structure not evidence of net trading profitability?

Only the causal interpretation has to recur. Identical rates were never the requirement.